# V15 - Operacao, suporte e escalonamento

**Objetivo (read-only):** produzir um diagnostico limitado e seguro para suporte: inventario, amostra e sinais simples que ajudam a triar um chamado.

In [ ]:
# Carrega a configuracao local e cria o cliente CDF. / Load local configuration and create the CDF client.
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials, Token

env_file = Path('.env')
pkg_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.example').exists()),
    Path.cwd(),
)

if env_file.exists():
    load_dotenv(env_file)
else:
    for name in ('.env', '.env.example'):
        candidate = pkg_root / name
        if candidate.exists():
            load_dotenv(candidate)
            break

if not os.environ.get('CDF_CLUSTER'):
    os.environ['CDF_CLUSTER'] = input('CDF_CLUSTER: ').strip()
if not os.environ.get('CDF_PROJECT'):
    os.environ['CDF_PROJECT'] = input('CDF_PROJECT: ').strip()

base_url = os.environ.get('CDF_URL', '').rstrip('/') or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
oauth_ready = env_file.exists() and all(
    os.environ.get(key) for key in ('IDP_TOKEN_URL', 'IDP_CLIENT_ID', 'IDP_CLIENT_SECRET')
)
if oauth_ready:
    scopes = os.environ.get('IDP_SCOPES', '').replace(',', ' ').split() or [f'{base_url}/.default']
    audience = os.environ.get('IDP_AUDIENCE', '')
    credentials = OAuthClientCredentials(
        token_url=os.environ['IDP_TOKEN_URL'],
        client_id=os.environ['IDP_CLIENT_ID'],
        client_secret=os.environ['IDP_CLIENT_SECRET'],
        scopes=scopes,
        **({'audience': audience} if audience else {}),
    )
else:
    # Sem .env nesta pasta: usa Bearer Token.
    bearer_token = (os.environ.get('CDF_BEARER_TOKEN') or getpass('CDF_BEARER_TOKEN: ')).strip()
    if bearer_token.lower().startswith('bearer '):
        bearer_token = bearer_token[7:].strip()
    credentials = Token(bearer_token)

client = CogniteClient(ClientConfig(
    client_name='radix-cdf-onboarding-v15',
    base_url=base_url,
    project=os.environ['CDF_PROJECT'],
    credentials=credentials,
))

## 1. Inventario rapido do ambiente

In [ ]:
inventory = {
    'spaces': len(client.data_modeling.spaces.list(limit=100)),
    'views': len(client.data_modeling.views.list(limit=100)),
    'data_models': len(client.data_modeling.data_models.list(limit=100)),
}
print('inventario para diagnostico:', inventory)

## 2. Amostra somente leitura
Uma amostra pequena de nodes para verificar acesso e formato.

In [ ]:
nodes = client.data_modeling.instances.list(instance_type='node', limit=10)
unique_ids = {(n.space, n.external_id) for n in nodes}
print('nodes na amostra:', len(nodes), '| chaves unicas:', len(unique_ids))
nodes.to_pandas().head()

## 3. Sinais por space
Contagem de nodes da amostra por space ajuda a localizar onde investigar.

In [ ]:
import pandas as pd

df = nodes.to_pandas()
if 'space' in df.columns and len(df):
    signals = df['space'].value_counts()
    print('distribuicao da amostra por space:')
    print(signals)
else:
    print('Amostra vazia: registrar no chamado que o ambiente nao retornou nodes acessiveis.')

## Checklist de escalonamento (para o ticket)
- Inventario coletado (spaces/views/data models)?
- Amostra acessivel e com chaves unicas?
- Mensagem de erro original anexada (sem token/secret)?
- Space/modelo suspeito identificado?

## Mini-exercicio
- Acrescente ao diagnostico a contagem de edges (`instance_type='edge'`).